# Brain Tumor Classification - Observability & Monitoring Demo

**Johns Hopkins University 635.603 - AI/ML Ops (Fall 2025)**

**Team Project:** Brain MRI Tumor Detection using CNNs

**Notebook Purpose:** Comprehensive demonstration of the observability layer for our ML pipeline

---

## Overview

This notebook demonstrates a complete, **architecture-agnostic** observability stack for monitoring machine learning models in production. All code works with ANY CNN architecture - from basic CNNs to transfer learning models.

### MLO Objectives Satisfied

This observability layer directly supports:

- **MLO 8.1** - Clear, modular ML pipeline with well-defined components
- **MLO 8.4** - Comprehensive tracking and metadata via MLflow
- **MLO 8.5** - Production-grade, reproducible ML code with quality checks
- **MLO 8.6** - Rigorous ML performance evaluation with clinical-grade metrics
- **MLO 8.7** - Deployment monitoring with drift detection and alerting
- **MLO 14.1** - Complete end-to-end demonstration for final presentation

### Final Demo Rubric Alignment

This notebook addresses all rubric requirements:

1. **Pipeline Clarity** - Modular observability components with clear responsibilities
2. **Algorithm Clarity** - Architecture-agnostic design works with any model
3. **Metrics Visualization** - Comprehensive metrics with clinical interpretation
4. **Deployment Demo** - Production monitoring simulation
5. **End-to-End Demo** - Complete workflow from training to production monitoring

### Project SMART Goals

Our observability layer enforces:
- **Recall ≥ 80%** - Alerts if we're missing too many tumors (clinical safety!)
- **Precision ≥ 80%** - Alerts if too many false positives
- **Production monitoring** - Continuous tracking to maintain these goals over time

---

## What This Notebook Demonstrates

1. **MLflow Experiment Tracking** - Reproducible experiment management
2. **Training Monitoring** - Epoch-by-epoch metric tracking
3. **Evaluation Metrics** - Clinical-grade performance assessment
4. **Data Quality Monitoring** - Image corruption and brightness checks
5. **Drift Detection** - Identifying when model or data changes
6. **Latency Monitoring** - Production performance tracking
7. **Automated Alerting** - Threshold-based alerts for critical issues
8. **End-to-End Monitoring** - Complete production monitoring simulation

---

## 1. Setup and Imports

First, let's import our observability modules and other required libraries.

In [ ]:
# Standard libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import time
import warnings
warnings.filterwarnings('ignore')

# MLflow for experiment tracking
import mlflow

# Our observability modules
import sys
sys.path.append('..')  # Add parent directory to path

from observability import mlflow_config
from observability import logger
from observability import metrics
from observability import data_quality
from observability import drift
from observability import alerts
from observability import monitoring

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ All imports successful!")
print("\nObservability modules loaded:")
print("  - mlflow_config: Experiment tracking infrastructure")
print("  - logger: Training and inference logging")
print("  - metrics: Performance evaluation")
print("  - data_quality: Image quality monitoring")
print("  - drift: Distribution shift detection")
print("  - alerts: Automated threshold-based alerting")
print("  - monitoring: End-to-end monitoring utilities")

## 2. MLflow Setup

MLflow provides experiment tracking, allowing us to:
- Log hyperparameters
- Track metrics across epochs
- Compare different runs
- Maintain reproducibility

**Note:** Currently using local file storage. Can easily migrate to remote server for team collaboration.

In [ ]:
# Setup MLflow with local tracking
experiment_id = mlflow_config.setup_mlflow()

# Print configuration
mlflow_config.print_mlflow_info()

print("\n💡 TIP: After running this notebook, you can view experiments in the MLflow UI:")
print("   Run in terminal: mlflow ui --backend-store-uri file:./mlruns")
print("   Then open: http://localhost:5000")

## 3. Simulated Training Demonstration

We'll simulate a training run to demonstrate metric tracking.
This shows how to integrate observability with ANY model architecture.

**Key Point:** This code works identically whether you're using:
- A basic CNN
- ResNet50 transfer learning
- EfficientNet
- Any custom architecture

In [ ]:
# Start a new MLflow run for this training simulation
mlflow_config.start_run(run_name="demo_training_simulation")

# Log training hyperparameters (architecture-agnostic!)
logger.log_training_params(
    learning_rate=0.001,
    batch_size=32,
    epochs=50,
    optimizer="adam",
    loss_function="binary_crossentropy",
    architecture="cnn_demo",  # Could be "resnet50", "custom_cnn", etc.
    additional_params={
        "dropout_rate": 0.3,
        "data_augmentation": True,
        "image_size": "224x224"
    }
)

print("✓ Logged training parameters to MLflow")

In [ ]:
# Simulate training over 50 epochs
print("\nSimulating training over 50 epochs...\n")

epochs = 50
training_history = {
    'epoch': [],
    'train_loss': [],
    'val_loss': [],
    'val_accuracy': [],
    'val_precision': [],
    'val_recall': [],
    'val_f1': []
}

# Simulate realistic learning curves
np.random.seed(42)

for epoch in range(epochs):
    # Simulate decreasing loss with some noise
    train_loss = 0.5 * np.exp(-epoch / 15) + 0.05 + np.random.normal(0, 0.01)
    val_loss = 0.5 * np.exp(-epoch / 15) + 0.08 + np.random.normal(0, 0.015)
    
    # Simulate increasing metrics with plateau
    progress = 1 - np.exp(-epoch / 10)
    val_accuracy = 0.70 + 0.18 * progress + np.random.normal(0, 0.01)
    val_precision = 0.72 + 0.16 * progress + np.random.normal(0, 0.01)
    val_recall = 0.68 + 0.20 * progress + np.random.normal(0, 0.015)
    val_f1 = 2 * val_precision * val_recall / (val_precision + val_recall)
    
    # Clip to valid ranges
    val_accuracy = np.clip(val_accuracy, 0, 1)
    val_precision = np.clip(val_precision, 0, 1)
    val_recall = np.clip(val_recall, 0, 1)
    val_f1 = np.clip(val_f1, 0, 1)
    
    # Log to MLflow
    logger.log_epoch_metrics(
        epoch=epoch,
        train_loss=train_loss,
        train_accuracy=0.70 + 0.20 * progress,  # Training usually better than val
        val_loss=val_loss,
        val_accuracy=val_accuracy,
        val_precision=val_precision,
        val_recall=val_recall,
        val_f1=val_f1
    )
    
    # Store for visualization
    training_history['epoch'].append(epoch)
    training_history['train_loss'].append(train_loss)
    training_history['val_loss'].append(val_loss)
    training_history['val_accuracy'].append(val_accuracy)
    training_history['val_precision'].append(val_precision)
    training_history['val_recall'].append(val_recall)
    training_history['val_f1'].append(val_f1)

print("✓ Completed 50 epochs of training simulation")
print(f"\nFinal Metrics:")
print(f"  Validation Accuracy:  {training_history['val_accuracy'][-1]:.2%}")
print(f"  Validation Precision: {training_history['val_precision'][-1]:.2%}")
print(f"  Validation Recall:    {training_history['val_recall'][-1]:.2%}")
print(f"  Validation F1:        {training_history['val_f1'][-1]:.2%}")

In [ ]:
# Visualize training history
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Loss curves
axes[0, 0].plot(training_history['epoch'], training_history['train_loss'], label='Train Loss', linewidth=2)
axes[0, 0].plot(training_history['epoch'], training_history['val_loss'], label='Val Loss', linewidth=2)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Training and Validation Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Accuracy
axes[0, 1].plot(training_history['epoch'], training_history['val_accuracy'], linewidth=2, color='green')
axes[0, 1].axhline(y=0.80, color='r', linestyle='--', label='Target: 80%', alpha=0.7)
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].set_title('Validation Accuracy')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Precision and Recall (CRITICAL for medical imaging)
axes[1, 0].plot(training_history['epoch'], training_history['val_precision'], label='Precision', linewidth=2)
axes[1, 0].plot(training_history['epoch'], training_history['val_recall'], label='Recall (CRITICAL!)', linewidth=2)
axes[1, 0].axhline(y=0.80, color='r', linestyle='--', label='Target: 80%', alpha=0.7)
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Score')
axes[1, 0].set_title('Precision and Recall (Clinical Safety Metrics)')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# F1 Score
axes[1, 1].plot(training_history['epoch'], training_history['val_f1'], linewidth=2, color='purple')
axes[1, 1].axhline(y=0.80, color='r', linestyle='--', label='Target: 80%', alpha=0.7)
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('F1 Score')
axes[1, 1].set_title('F1 Score (Balanced Performance)')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Training visualization saved as 'training_history.png'")

In [ ]:
# Log training summary
training_time = 1850  # Simulated training time in seconds

logger.log_training_summary(
    total_epochs_completed=epochs,
    best_val_accuracy=max(training_history['val_accuracy']),
    best_val_recall=max(training_history['val_recall']),
    best_val_precision=max(training_history['val_precision']),
    final_train_loss=training_history['train_loss'][-1],
    training_time_seconds=training_time
)

# End this MLflow run
mlflow_config.end_run()

print("\n✓ Training run logged to MLflow successfully!")

## 4. Data Quality Monitoring

Before evaluating the model, let's demonstrate data quality checks.
In production, these would run on incoming MRI images to detect:
- Corrupted files
- Unusual brightness (scanner calibration issues)
- Distribution shifts

**Note:** We'll simulate data since we're demonstrating the observability infrastructure.

In [ ]:
# Simulate image brightness data for a dataset
print("Simulating data quality analysis...\n")

# Simulate brightness values for a healthy dataset
np.random.seed(42)
num_images = 500

# Most images have normal brightness (centered around 120)
brightness_values = np.random.normal(120, 25, num_images)

# Add some outliers (very dark or very bright images)
num_outliers = 15
outlier_indices = np.random.choice(num_images, num_outliers, replace=False)
brightness_values[outlier_indices[:8]] = np.random.uniform(10, 30, 8)  # Very dark
brightness_values[outlier_indices[8:]] = np.random.uniform(230, 250, 7)  # Very bright

# Clip to valid range
brightness_values = np.clip(brightness_values, 0, 255)

# Simulate corruption (would be detected by data_quality.detect_corrupted_image in production)
num_corrupted = 12
corruption_rate = num_corrupted / num_images

# Create a quality report (simulated)
simulated_quality_report = {
    'total_images': num_images,
    'valid_images': num_images - num_corrupted,
    'corrupted_count': num_corrupted,
    'corrupted_percentage': corruption_rate * 100,
    'brightness_mean': float(np.mean(brightness_values)),
    'brightness_std': float(np.std(brightness_values)),
    'brightness_min': float(np.min(brightness_values)),
    'brightness_max': float(np.max(brightness_values)),
    'brightness_median': float(np.median(brightness_values)),
    'brightness_values': brightness_values.tolist(),
}

# Print quality report
data_quality.print_quality_report(simulated_quality_report)

In [ ]:
# Visualize brightness distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(brightness_values, bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(simulated_quality_report['brightness_mean'], 
                color='r', linestyle='--', linewidth=2, label=f"Mean: {simulated_quality_report['brightness_mean']:.1f}")
axes[0].axvline(20, color='orange', linestyle=':', linewidth=2, label='Suspicious threshold (too dark)')
axes[0].axvline(240, color='orange', linestyle=':', linewidth=2, label='Suspicious threshold (too bright)')
axes[0].set_xlabel('Brightness (0-255)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Image Brightness Distribution')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Box plot
axes[1].boxplot(brightness_values, vert=True)
axes[1].set_ylabel('Brightness (0-255)')
axes[1].set_title('Brightness Box Plot (Shows Outliers)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('data_quality_brightness.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Data quality visualization saved as 'data_quality_brightness.png'")

In [ ]:
# Check for data quality alerts
quality_alerts = alerts.check_data_quality_alerts(simulated_quality_report)

if quality_alerts:
    print("\n⚠️  Data Quality Alerts Detected:\n")
    for alert in quality_alerts:
        print(f"  {alert}")
else:
    print("\n✓ No data quality alerts - dataset looks good!")

## 5. Model Performance Evaluation

Now let's demonstrate comprehensive model evaluation.
This section shows how to:
- Generate predictions (simulated)
- Compute all evaluation metrics
- Visualize results
- Check for performance alerts

**Clinical Context:** For brain tumor detection:
- **Recall** is MOST CRITICAL (missing a tumor can be fatal)
- **Precision** is important (false positives cause unnecessary procedures/anxiety)
- Both should be ≥ 80% per project goals

In [ ]:
# Generate simulated test predictions
print("Generating simulated model predictions on test set...\n")

y_true, y_pred, y_prob = monitoring.generate_simulated_predictions(
    num_samples=800,
    true_positive_rate=0.30,  # 30% of test cases have tumors
    model_recall=0.88,        # Model catches 88% of tumors
    model_precision=0.85      # 85% of tumor predictions are correct
)

print(f"Generated {len(y_true)} test predictions")
print(f"True positive cases (tumors): {np.sum(y_true)} ({np.mean(y_true):.1%})")
print(f"Predicted positive (tumors): {np.sum(y_pred)} ({np.mean(y_pred):.1%})")

In [ ]:
# Compute comprehensive metrics
print("\nComputing classification metrics...\n")

test_metrics = metrics.compute_classification_metrics(y_true, y_pred, y_prob)

# Print detailed metrics report
metrics.print_metrics(test_metrics, show_interpretation=True, show_confusion_matrix=True)

In [ ]:
# Visualize confusion matrix
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True,
            xticklabels=['No Tumor', 'Tumor'],
            yticklabels=['No Tumor', 'Tumor'],
            ax=ax)
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')
ax.set_title('Confusion Matrix\n\n'
             'CRITICAL: False Negatives (bottom-left) = Missed Tumors!\n'
             'IMPORTANT: False Positives (top-right) = Unnecessary Procedures',
             fontsize=12)

# Add annotations
tn, fp, fn, tp = cm.ravel()
ax.text(0.5, -0.15, f'False Negatives (Missed Tumors): {fn}',
        ha='center', transform=ax.transAxes, fontsize=10, color='red', weight='bold')

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Confusion matrix saved as 'confusion_matrix.png'")

In [ ]:
# Visualize ROC curve (if probabilities available)
from sklearn.metrics import roc_curve, auc

fpr, tpr, thresholds = roc_curve(y_true, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, 
         label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curve - Model Discrimination Ability')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ ROC curve saved as 'roc_curve.png'")

In [ ]:
# Check for performance alerts
performance_alerts = alerts.check_performance_alerts(test_metrics)

if performance_alerts:
    print("\n⚠️  PERFORMANCE ALERTS DETECTED:\n")
    for alert in performance_alerts:
        print(f"  {alert}")
else:
    print("\n✓ No performance alerts - model meets all thresholds!")

## 6. Drift Detection

Drift occurs when model predictions or input data change over time.
This is critical for production monitoring:
- **Prediction Drift**: Model outputs change (e.g., predicting more/fewer tumors)
- **Embedding Drift**: Input characteristics change (e.g., different scanner)

Let's simulate drift detection by comparing baseline (training) to production data.

In [ ]:
# Generate baseline predictions (from training/validation)
print("Simulating baseline (reference) predictions...\n")

baseline_y_true, baseline_y_pred, baseline_y_prob = monitoring.generate_simulated_predictions(
    num_samples=1000,
    true_positive_rate=0.30,
    model_recall=0.88,
    model_precision=0.85
)

print(f"Baseline positive rate: {np.mean(baseline_y_pred):.2%}")

# Generate "production" predictions with drift
print("\nSimulating production predictions (with drift)...\n")

production_y_true, production_y_pred, production_y_prob = monitoring.generate_simulated_predictions(
    num_samples=500,
    true_positive_rate=0.40,  # Different patient population (more high-risk)
    model_recall=0.82,         # Slightly degraded
    model_precision=0.80       # Slightly degraded
)

print(f"Production positive rate: {np.mean(production_y_pred):.2%}")

In [ ]:
# Detect drift
print("\nRunning drift detection...\n")

drift_report = drift.monitor_drift(
    reference_predictions=baseline_y_pred,
    current_predictions=production_y_pred,
    # Note: Embedding drift requires feature vectors from model
    # We'll skip it for now, but it's easy to add later!
)

# Print drift report
drift.print_drift_report(drift_report)

In [ ]:
# Visualize drift
baseline_rate = drift_report['prediction_drift']['reference_positive_rate']
production_rate = drift_report['prediction_drift']['current_positive_rate']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart comparing positive rates
periods = ['Baseline', 'Production']
rates = [baseline_rate, production_rate]
colors = ['green' if drift_report['prediction_drift']['severity'] == 'none' else 'orange']
colors = ['green', 'orange']

axes[0].bar(periods, rates, color=colors, alpha=0.7, edgecolor='black')
axes[0].set_ylabel('Positive Prediction Rate')
axes[0].set_title('Prediction Drift: Positive Rate Comparison')
axes[0].set_ylim([0, max(rates) * 1.2])
for i, (period, rate) in enumerate(zip(periods, rates)):
    axes[0].text(i, rate + 0.01, f'{rate:.2%}', ha='center', fontsize=12, weight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

# Probability distribution comparison
axes[1].hist(baseline_y_prob, bins=30, alpha=0.5, label='Baseline', edgecolor='black')
axes[1].hist(production_y_prob, bins=30, alpha=0.5, label='Production', edgecolor='black')
axes[1].set_xlabel('Prediction Probability')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Probability Distribution Shift')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('drift_detection.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Drift visualization saved as 'drift_detection.png'")

In [ ]:
# Check for drift alerts
drift_alerts = alerts.check_drift_alerts(drift_report)

if drift_alerts:
    print("\n⚠️  DRIFT ALERTS DETECTED:\n")
    for alert in drift_alerts:
        print(f"  {alert}")
else:
    print("\n✓ No drift alerts - distributions are stable")

## 7. Latency Monitoring

In production, we must monitor inference latency to ensure:
- Acceptable response times for users
- No performance degradation
- Early detection of infrastructure issues

Let's simulate latency monitoring.

In [ ]:
# Simulate latency measurements
print("Simulating 1000 inference requests...\n")

latencies = monitoring.simulate_latency_samples(
    num_samples=1000,
    mean_ms=180,
    std_ms=35,
    add_outliers=True,
    outlier_probability=0.03  # 3% of requests are slow
)

# Compute statistics
latency_stats = monitoring.compute_latency_stats(latencies)

print("Latency Statistics:")
print(f"  Mean:     {latency_stats['mean']:.1f} ms")
print(f"  Median:   {latency_stats['median']:.1f} ms")
print(f"  Std Dev:  {latency_stats['std']:.1f} ms")
print(f"  Min:      {latency_stats['min']:.1f} ms")
print(f"  Max:      {latency_stats['max']:.1f} ms")
print(f"  p50:      {latency_stats['p50']:.1f} ms")
print(f"  p90:      {latency_stats['p90']:.1f} ms")
print(f"  p95:      {latency_stats['p95']:.1f} ms")
print(f"  p99:      {latency_stats['p99']:.1f} ms")

In [ ]:
# Visualize latency distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(latencies, bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(latency_stats['mean'], color='r', linestyle='--', linewidth=2,
                label=f"Mean: {latency_stats['mean']:.1f}ms")
axes[0].axvline(latency_stats['p95'], color='orange', linestyle='--', linewidth=2,
                label=f"p95: {latency_stats['p95']:.1f}ms")
axes[0].set_xlabel('Latency (ms)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Inference Latency Distribution')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Percentile plot
percentiles = np.arange(0, 101, 1)
latency_percentiles = np.percentile(latencies, percentiles)

axes[1].plot(percentiles, latency_percentiles, linewidth=2)
axes[1].axhline(y=1000, color='r', linestyle='--', linewidth=2, 
                label='Alert threshold (1000ms)', alpha=0.7)
axes[1].scatter([95], [latency_stats['p95']], color='orange', s=100, zorder=5,
                label=f"p95: {latency_stats['p95']:.1f}ms")
axes[1].set_xlabel('Percentile')
axes[1].set_ylabel('Latency (ms)')
axes[1].set_title('Latency Percentiles')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('latency_monitoring.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Latency visualization saved as 'latency_monitoring.png'")

In [ ]:
# Check for latency alerts
latency_alerts = alerts.check_latency_alerts(latency_stats)

if latency_alerts:
    print("\n⚠️  LATENCY ALERTS DETECTED:\n")
    for alert in latency_alerts:
        print(f"  {alert}")
else:
    print("\n✓ No latency alerts - performance is good")

## 8. Comprehensive Alert Summary

Let's collect all alerts from different monitoring components and create a comprehensive alert dashboard.

In [ ]:
# Collect all alerts
all_alerts = alerts.check_all_alerts(
    metrics=test_metrics,
    drift_report=drift_report,
    quality_report=simulated_quality_report,
    latency_stats=latency_stats
)

# Print comprehensive alert summary
alerts.print_alerts(all_alerts, show_metadata=True)

# Save alerts to file for audit trail
alerts.save_alerts(all_alerts, 'alerts_report.json')

In [ ]:
# Alert summary visualization
if all_alerts:
    # Count alerts by type and severity
    alert_counts = {'performance': 0, 'drift': 0, 'quality': 0, 'operational': 0}
    severity_counts = {'critical': 0, 'high': 0, 'warning': 0, 'info': 0}
    
    for alert in all_alerts:
        alert_counts[alert.alert_type] = alert_counts.get(alert.alert_type, 0) + 1
        severity_counts[alert.severity] = severity_counts.get(alert.severity, 0) + 1
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Alerts by type
    types = list(alert_counts.keys())
    counts = list(alert_counts.values())
    axes[0].bar(types, counts, color=['#ff6b6b', '#4ecdc4', '#45b7d1', '#96ceb4'], 
                edgecolor='black', alpha=0.7)
    axes[0].set_ylabel('Number of Alerts')
    axes[0].set_title('Alerts by Type')
    axes[0].grid(True, alpha=0.3, axis='y')
    
    # Alerts by severity
    severities = ['critical', 'high', 'warning', 'info']
    sev_counts = [severity_counts.get(s, 0) for s in severities]
    colors = ['#d9534f', '#f0ad4e', '#f7b733', '#5bc0de']
    axes[1].bar(severities, sev_counts, color=colors, edgecolor='black', alpha=0.7)
    axes[1].set_ylabel('Number of Alerts')
    axes[1].set_title('Alerts by Severity')
    axes[1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig('alert_summary.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\n✓ Alert summary saved as 'alert_summary.png'")
else:
    print("\n✓ No alerts to visualize - all systems normal!")

## 9. Complete End-to-End Monitoring Demo

Finally, let's run a complete monitoring demonstration that showcases all components working together.
This simulates a real production monitoring scenario.

In [ ]:
# Run comprehensive monitoring demo
demo_results = monitoring.run_monitoring_demo(verbose=True)

In [ ]:
# Create a summary dashboard
print("\n" + "=" * 70)
print("OBSERVABILITY STACK SUMMARY")
print("=" * 70)

print("\n📊 COMPONENTS DEMONSTRATED:\n")
print("  ✓ MLflow experiment tracking")
print("  ✓ Training metric logging (50 epochs)")
print("  ✓ Data quality monitoring (brightness, corruption)")
print("  ✓ Classification metrics (precision, recall, F1, ROC-AUC)")
print("  ✓ Drift detection (prediction distribution)")
print("  ✓ Latency monitoring (p95, p99, outliers)")
print("  ✓ Automated alerting (threshold-based)")
print("  ✓ End-to-end monitoring pipeline")

print("\n🎯 PROJECT GOALS STATUS:\n")
recall_met = test_metrics['recall'] >= 0.80
precision_met = test_metrics['precision'] >= 0.80
print(f"  {'✓' if recall_met else '✗'} Recall ≥ 80%: {test_metrics['recall']:.2%}")
print(f"  {'✓' if precision_met else '✗'} Precision ≥ 80%: {test_metrics['precision']:.2%}")

if recall_met and precision_met:
    print("\n  🎉 BOTH SMART GOALS ACHIEVED!")
else:
    print("\n  ⚠️  Goals not met - model needs improvement")

print("\n📈 MLO OBJECTIVES SATISFIED:\n")
print("  ✓ MLO 8.1  - Clear, modular ML pipeline")
print("  ✓ MLO 8.4  - Tracking and metadata via MLflow")
print("  ✓ MLO 8.5  - Production-grade, reproducible code")
print("  ✓ MLO 8.6  - Rigorous ML performance evaluation")
print("  ✓ MLO 8.7  - Deployment monitoring capabilities")
print("  ✓ MLO 14.1 - Complete end-to-end demonstration")

print("\n🏥 CLINICAL SAFETY FEATURES:\n")
print("  ✓ Recall-focused metrics (minimize missed tumors)")
print("  ✓ Automated alerts for low recall (<80%)")
print("  ✓ Drift detection (catches distribution shifts)")
print("  ✓ Data quality checks (corrupted images)")
print("  ✓ Latency monitoring (ensure timely diagnoses)")

print("\n🔧 ARCHITECTURE-AGNOSTIC DESIGN:\n")
print("  ✓ Works with ANY CNN architecture")
print("  ✓ No hard-coded model assumptions")
print("  ✓ Plug-and-play with team's chosen model")
print("  ✓ Easy to extend and customize")

print("\n📁 GENERATED ARTIFACTS:\n")
artifacts = [
    'training_history.png',
    'data_quality_brightness.png',
    'confusion_matrix.png',
    'roc_curve.png',
    'drift_detection.png',
    'latency_monitoring.png',
    'alert_summary.png',
    'alerts_report.json'
]
for artifact in artifacts:
    print(f"  • {artifact}")

print("\n💡 NEXT STEPS:\n")
print("  1. View MLflow UI: mlflow ui --backend-store-uri file:./mlruns")
print("  2. Integrate with actual model training code")
print("  3. Deploy monitoring to production environment")
print("  4. Set up automated alert notifications (email/Slack)")
print("  5. Create monitoring dashboard for continuous oversight")

print("\n" + "=" * 70)
print("DEMONSTRATION COMPLETE ✓")
print("=" * 70 + "\n")

## 10. Final Summary

### What We've Demonstrated

This notebook has shown a **complete, production-ready observability stack** for ML model monitoring:

#### 1. Experiment Tracking (MLflow)
- Reproducible experiment management
- Parameter and metric logging
- Easy comparison of runs

#### 2. Training Monitoring
- Epoch-by-epoch metric tracking
- Learning curve visualization
- Training summary statistics

#### 3. Data Quality
- Image brightness monitoring
- Corruption detection
- Dataset distribution analysis

#### 4. Model Evaluation
- Comprehensive metrics (accuracy, precision, recall, F1, ROC-AUC)
- Clinical interpretation
- Confusion matrix analysis
- Goal assessment (≥80% precision and recall)

#### 5. Drift Detection
- Prediction distribution monitoring
- Statistical tests for drift
- Optional embedding-based drift (when available)

#### 6. Latency Monitoring
- Response time tracking
- Percentile analysis (p95, p99)
- Outlier detection

#### 7. Automated Alerting
- Threshold-based alerts
- Multiple severity levels
- Actionable recommendations

---

### Key Advantages

**✅ Architecture-Agnostic**
- Works with any CNN model
- No hard-coded assumptions
- Team can choose any architecture

**✅ Production-Ready**
- Comprehensive error handling
- Configurable thresholds
- Audit trail via MLflow and JSON logs

**✅ Clinically Aware**
- Recall prioritization (patient safety)
- Clear interpretation of metrics
- Alert thresholds based on clinical requirements

**✅ Well-Documented**
- Extensive inline comments
- Clear docstrings
- Usage examples

---

### Connection to Course Objectives

This observability layer directly supports:

| MLO | Requirement | How We Address It |
|-----|-------------|-------------------|
| **8.1** | Clear ML pipeline | Modular components with defined interfaces |
| **8.4** | Tracking/metadata | MLflow experiment tracking |
| **8.5** | Production-grade code | Error handling, logging, documentation |
| **8.6** | Performance evaluation | Comprehensive metrics with interpretation |
| **8.7** | Deployment monitoring | Drift, quality, latency monitoring |
| **14.1** | Final demonstration | This complete notebook! |

---

### Ready for Final Presentation

This notebook and the underlying modules provide everything needed for the final demo:

1. ✅ **Pipeline Clarity** - Each module has a clear purpose
2. ✅ **Algorithm Clarity** - Architecture-agnostic design
3. ✅ **Metrics Visualization** - Multiple plots and dashboards
4. ✅ **Deployment Demo** - Production monitoring simulation
5. ✅ **Complete Demo** - End-to-end workflow shown

---

### Questions for Review

When presenting or reviewing this work, consider:

1. **Why is recall more important than precision for tumor detection?**
   - Missing a tumor (false negative) can be fatal
   - False positive leads to extra scan, not life-threatening

2. **Why is drift detection important?**
   - Data changes over time (new scanners, populations)
   - Model may degrade without retraining
   - Early warning prevents poor predictions

3. **How does this work with different CNN architectures?**
   - All functions use generic inputs (y_true, y_pred, y_prob)
   - No assumptions about internal model structure
   - Works identically for basic CNN, ResNet, EfficientNet, etc.

4. **What happens in production deployment?**
   - MLflow tracks all predictions
   - Drift detection runs weekly/daily
   - Alerts sent to team (email/Slack)
   - Dashboard shows current system health

---

### Acknowledgments

**Course:** Johns Hopkins 635.603 - AI/ML Ops (Fall 2025)

**Project:** Brain MRI Tumor Classification

**Dataset:** Brain MRI Images for Brain Tumor Detection (Kaggle)

**Technologies:**
- MLflow (experiment tracking)
- scikit-learn (metrics)
- NumPy/Pandas (data manipulation)
- Matplotlib/Seaborn (visualization)

---

**END OF NOTEBOOK**